# Phase 5: Geographic and Store Analysis

This notebook analyzes country performance, store performance, store rankings, and store growth trends.

## 1. Import libraries and load the cleaned dataset

In [ ]:
from pathlib import Path
import pandas as pd

current_folder = Path.cwd()

if current_folder.name == 'notebooks':
    project_folder = current_folder.parent
else:
    project_folder = current_folder

cleaned_file = project_folder / 'data' / 'processed' / 'cleaned_sales.csv'
sales = pd.read_csv(cleaned_file)

print('Rows loaded:', len(sales))

## 2. Analyze country performance

Country performance is based on the customer's country.

In [ ]:
country_performance = (
    sales.groupby('Country_Customer', as_index=False)
    .agg(
        Revenue=('Sales USD', 'sum'),
        Profit=('Profit USD', 'sum'),
        Customers=('CustomerKey', 'nunique'),
        Orders=('Order Number', 'nunique'),
    )
    .rename(columns={'Country_Customer': 'Country'})
)

country_performance['Profit Margin %'] = (
    country_performance['Profit']
    / country_performance['Revenue']
    * 100
)

country_performance[['Revenue', 'Profit', 'Profit Margin %']] = (
    country_performance[['Revenue', 'Profit', 'Profit Margin %']].round(2)
)

country_performance = country_performance.sort_values('Revenue', ascending=False)
display(country_performance)

## 3. Answer the country questions

In [ ]:
best_country = country_performance.loc[country_performance['Revenue'].idxmax()]
worst_country = country_performance.loc[country_performance['Revenue'].idxmin()]
highest_margin_country = country_performance.loc[country_performance['Profit Margin %'].idxmax()]

print('Best-performing country:', best_country['Country'])
print(f'Revenue: ${best_country["Revenue"]:,.2f}')
print()
print('Worst-performing country:', worst_country['Country'])
print(f'Revenue: ${worst_country["Revenue"]:,.2f}')
print()
print('Country with highest profit margin:', highest_margin_country['Country'])
print(f'Profit margin: {highest_margin_country["Profit Margin %"]:.2f}%')

## 4. Analyze store performance

In [ ]:
store_performance = (
    sales.groupby(['StoreKey', 'Country_Store', 'State_Store'], as_index=False)
    .agg(
        Revenue=('Sales USD', 'sum'),
        Profit=('Profit USD', 'sum'),
        Orders=('Order Number', 'nunique'),
    )
)

store_performance['Profit Margin %'] = (
    store_performance['Profit']
    / store_performance['Revenue']
    * 100
)

store_performance[['Revenue', 'Profit', 'Profit Margin %']] = (
    store_performance[['Revenue', 'Profit', 'Profit Margin %']].round(2)
)

store_performance = store_performance.sort_values('Revenue', ascending=False)
display(store_performance)

## 5. Find the top 10 stores

In [ ]:
top_10_stores = store_performance.head(10)
display(top_10_stores)

## 6. Find the 10 lowest-performing stores

In [ ]:
lowest_10_stores = store_performance.tail(10).sort_values('Revenue')
display(lowest_10_stores)

## 7. Analyze yearly store growth

In [ ]:
store_growth = (
    sales.groupby(['StoreKey', 'Country_Store', 'State_Store', 'Year'], as_index=False)
    .agg(Revenue=('Sales USD', 'sum'))
    .sort_values(['StoreKey', 'Year'])
)

store_growth['Previous Year Revenue'] = (
    store_growth.groupby('StoreKey')['Revenue'].shift(1)
)

store_growth['Growth %'] = (
    (store_growth['Revenue'] - store_growth['Previous Year Revenue'])
    / store_growth['Previous Year Revenue']
    * 100
)

store_growth[['Revenue', 'Previous Year Revenue', 'Growth %']] = (
    store_growth[['Revenue', 'Previous Year Revenue', 'Growth %']].round(2)
)

display(store_growth)

## 8. Save the country and store datasets

In [ ]:
processed_folder = project_folder / 'data' / 'processed'
country_file = processed_folder / 'country_performance.csv'
store_file = processed_folder / 'store_performance.csv'
store_growth_file = processed_folder / 'store_growth.csv'

country_performance.to_csv(country_file, index=False)
store_performance.to_csv(store_file, index=False)
store_growth.to_csv(store_growth_file, index=False)

print('Saved:', country_file)
print('Saved:', store_file)
print('Saved:', store_growth_file)

## 9. Display the Phase 5 answers

In [ ]:
print('PHASE 5 ANSWERS')
print('-' * 50)
print(f'Best-performing country: {best_country["Country"]}')
print(f'Worst-performing country: {worst_country["Country"]}')
print(f'Country with highest profit margin: {highest_margin_country["Country"]}')
print(f'Top store: {int(top_10_stores.iloc[0]["StoreKey"])}')
print(f'Lowest-performing store: {int(lowest_10_stores.iloc[0]["StoreKey"])}')
print(f'Top store revenue: ${top_10_stores.iloc[0]["Revenue"]:,.2f}')
print(f'Lowest store revenue: ${lowest_10_stores.iloc[0]["Revenue"]:,.2f}')